# Historial de la muestra

##### En la primer parte descargamos las features históricas para la muestra seleccionada del snap 28. En la segunda parte calculamos el historial de mergers.

In [9]:
import src.merger_tree_tools as mtt
import numpy as np
from astropy.table import Table
from astropy.cosmology import Planck15 as cosmo
import h5py
import pandas as pd
import polars as pl

In [10]:
# Ruta de los datos
path_lin ='data/' 

In [11]:
# Cargamos los datos de las galaxias centrales del snapshot 28 (z=0.1) para obtener los GalaxyID de cada una de ellas.

# Para la simu RefL0100N1504, el snapshot 28.
#df  = pl.read_csv(path_lin+'data_centrales_snap28.csv')

# Para la simu NoAGNL0050N0752, el snapshot 28.

df  = pl.read_csv(path_lin+'data_centrales_NoAGNL0050N0752_snap28.csv')

In [12]:
# Primeras 10 gal de la muestra
df.head(10)

GalaxyID,GroupID,SnapNum,SubGroupNumber,Stars_Mass,BlackHoleMass,SF_Mass,SF_Oxygen,SF_Hydrogen,metalicidad
i64,i64,i64,i64,f64,f64,f64,f64,f64,f64
1620592,28000000000014,28,0,2.7566e11,0.0,3.2630e9,0.03366,0.553539,9.579848
3126435,28000000000135,28,0,3.9125e10,0.0,6.0119e9,0.014139,0.675768,9.116511
3131127,28000000000136,28,0,1.3176e10,0.0,1.7319e9,0.014267,0.674644,9.121141
3136043,28000000000137,28,0,4.2697e10,0.0,7.8362e9,0.016327,0.664473,9.186315
3139793,28000000000138,28,0,4.4420e10,0.0,1.1540e10,0.014467,0.676644,9.125908
3143795,28000000000139,28,0,5.2209e10,0.0,6.9704e9,0.016568,0.662127,9.19422
3147527,28000000000140,28,0,6.7999e10,0.0,6.8181e9,0.020413,0.637047,9.301617
3150811,28000000000141,28,0,4.6484e10,0.0,2.7393e9,0.022646,0.622643,9.356628
3154894,28000000000142,28,0,4.0736e10,0.0,7.2762e9,0.015496,0.669099,9.160606


In [13]:
# Longitud de la muestra
len(df)

320

### Etapa de variables históricas

In [14]:
# Simulación y snapshot a analizar
simu='NoAGNL0050N0752'
snap=28

# Lista de variables a descargar.

columns=[
          'GalaxyID', 'SnapNum','Redshift',
          # Propiedades de las estrellas
          'Stars_Mass','Stars_KineticEnergy','Stars_Spin_x','Stars_Spin_y','Stars_Spin_z',
          # Propiedades SF
          'SF_Mass','SF_Oxygen','SF_Hydrogen','SF_MassWeightedTemperature','SF_KineticEnergy',
          'SF_ThermalEnergy','SF_TotalEnergy','SF_Spin_x','SF_Spin_y','SF_Spin_z',
          # Propiedades de los Agujeros Negros
          #'BlackHoleMass','BlackHoleMassAccretionRate'
        ]

In [15]:
long = len(df['GalaxyID'])

#long = 2

# Usuario y contraseña para conectarse a EAGLE DataBase
usr='cht015'
pwd='BH457tfj'

# Descargar merger tree completo de la galaxia deseada
# Nombre y alias de la tabla de la cual se quieren descargar datos
table='SubHalo'
table_alias='sub'


tracks = []

#Loop para todas las galaxias de la tabla
for p in np.arange(long):

    galid = df['GalaxyID'].to_numpy()[p]
 
    # Descargar todos los IDS necesarios de la galaxia deseada

    myIDs    = mtt.retrieve_ids(usr,pwd,simu,snap,galid)

    raw_tree = mtt.download_merger_tree(usr,pwd,simu,myIDs['GalaxyID'],myIDs['LastProgID'],
                                  table=table,table_alias=table_alias,columns=columns)
    
    # Aplicar condiciones a las galaxias del árbol, si es necesario
    mask = (
            (raw_tree['Stars_Mass']>0)&(raw_tree['SnapNum']>11) #& (raw_tree['Redshift']<2)
           )

    # Armar arbol sólo con galaxias seleccionadas según condiciones anteriores
    tree = {}
    for key in raw_tree.keys():
        tree[key]=raw_tree[key][mask]
        
    # Ordeno las galaxias según SnapNum creciente
    mask_order = (tree['SnapNum']).argsort()
    for key in tree.keys():
        tree[key] = tree[key][mask_order]

    # Armo un diccionario con solo la main branch del árbol
    main_branch = {}
    # Select galaxies in the main branch 
    mask_main = np.logical_and(tree['GalaxyID']>=myIDs['GalaxyID'],
                               tree['GalaxyID']<=myIDs['TopLeafID'])
    
    
    tmp = {
        key: tree[key][mask_main]
        for key in tree.keys()

    }
    tmp['RootGalaxyID'] = np.repeat(galid, len(tmp['GalaxyID']))

    tracks.append(pl.DataFrame(tmp))


In [16]:
# DataFrame final con la historia evolutiva de las galaxias centrales

tracks = pl.concat(tracks)

In [17]:
tracks.shape

(5417, 20)

In [18]:
# Guardamos el DataFrame con la historia evolutiva de las galaxias centrales
tracks.write_csv(path_lin+'tracks_centrales_NoAGNL0050N0752_snap28.csv')

In [19]:
tracks

GalaxyID,SnapNum,Redshift,Stars_Mass,Stars_KineticEnergy,Stars_Spin_x,Stars_Spin_y,Stars_Spin_z,SF_Mass,SF_Oxygen,SF_Hydrogen,SF_MassWeightedTemperature,SF_KineticEnergy,SF_ThermalEnergy,SF_TotalEnergy,SF_Spin_x,SF_Spin_y,SF_Spin_z,SubHaloGalaxyID,RootGalaxyID
i64,i32,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i64,i64
1620608,12,3.016505,4.6549e9,4.7754e13,-110.170067,-113.38382,-1.597729,9.9129e9,0.004171,0.736952,14096.604492,1.1861e14,2.6294e12,-1.0474e15,-11.874759,-203.963608,-355.954803,1620608,1620592
1620607,13,2.478413,8.9888e9,2.2588e14,-179.573059,269.90918,186.756577,1.6828e10,0.004805,0.73332,19545.839844,4.3473e14,6.3381e12,-2.7079e15,-137.959,-132.179214,269.664978,1620607,1620592
1620606,14,2.237037,1.4998e10,3.2201e14,-257.947205,83.829468,216.859085,1.8747e10,0.007965,0.719363,26476.669922,3.7830e14,9.6395e12,-3.8851e15,-546.21814,12.207579,289.370514,1620606,1620592
1620605,15,2.01241,2.0967e10,4.6505e14,-299.388336,122.416832,257.01239,1.6859e10,0.011262,0.701538,27409.744141,3.6749e14,8.9570e12,-3.6810e15,-598.541565,236.10997,243.310715,1620605,1620592
1620604,16,1.736966,2.6801e10,5.8635e14,-276.648529,223.813751,194.021408,1.5091e10,0.013725,0.684758,19762.441406,3.8680e14,5.7172e12,-2.9602e15,-725.208069,723.841736,366.282288,1620604,1620592
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3265057,24,0.365669,1.0505e10,8.4338e13,447.775787,143.798538,-1554.364624,6.4188e9,0.008075,0.714317,6997.230957,6.0414e13,8.1927e11,-3.5340e14,722.305359,118.801414,-2687.99292,3265057,3265053
3265056,25,0.270901,1.1977e10,1.0469e14,328.486603,131.899368,-1642.179077,7.3717e9,0.00804,0.714627,6557.0,8.1380e13,8.7856e11,-4.1497e14,522.453491,25.953718,-3046.803955,3265056,3265053
3265055,26,0.18271,1.3769e10,1.2435e14,185.880539,115.280823,-2014.611816,6.2686e9,0.009385,0.707758,5991.151367,7.0693e13,7.0782e11,-3.8186e14,253.538284,57.120804,-3069.684082,3265055,3265053


In [32]:
lookbacktime = [0,1.35,2.32,3.24,4.12,5.22,5.98,6.69,7.35,7.96,8.86,9.49,10.05,10.54,10.87,11.17,11.68]
for i in range(len(lookbacktime)-1):
    print(f'Entre {lookbacktime[i]} y {lookbacktime[i+1]} Gyr hay {-lookbacktime[i]+lookbacktime[i+1]} Gyr')

Entre 0 y 1.35 Gyr hay 1.35 Gyr
Entre 1.35 y 2.32 Gyr hay 0.9699999999999998 Gyr
Entre 2.32 y 3.24 Gyr hay 0.9200000000000004 Gyr
Entre 3.24 y 4.12 Gyr hay 0.8799999999999999 Gyr
Entre 4.12 y 5.22 Gyr hay 1.0999999999999996 Gyr
Entre 5.22 y 5.98 Gyr hay 0.7600000000000007 Gyr
Entre 5.98 y 6.69 Gyr hay 0.71 Gyr
Entre 6.69 y 7.35 Gyr hay 0.6599999999999993 Gyr
Entre 7.35 y 7.96 Gyr hay 0.6100000000000003 Gyr
Entre 7.96 y 8.86 Gyr hay 0.8999999999999995 Gyr
Entre 8.86 y 9.49 Gyr hay 0.6300000000000008 Gyr
Entre 9.49 y 10.05 Gyr hay 0.5600000000000005 Gyr
Entre 10.05 y 10.54 Gyr hay 0.48999999999999844 Gyr
Entre 10.54 y 10.87 Gyr hay 0.33000000000000007 Gyr
Entre 10.87 y 11.17 Gyr hay 0.3000000000000007 Gyr
Entre 11.17 y 11.68 Gyr hay 0.5099999999999998 Gyr


### Analizamos el historial de merger de las galxias.

In [20]:
# Simulación y snapshot a analizar
simu='NoAGNL0050N0752'
snap=28

In [21]:
# Va a contener el historial de fusiones
rows = []

In [22]:
long = len(df['GalaxyID'])

#long = 5

# Usuario y contraseña para conectarse a EAGLE DataBase
usr='cht015'
pwd='BH457tfj'

# Descargar merger tree completo de la galaxia deseada
# Nombre y alias de la tabla de la cual se quieren descargar datos
table='SubHalo'
table_alias='sub'

# Variables que se quiere descargar. OJO!! Asegurarse que estas variables
# estén en la tabla deseada.
columns=[
          'GalaxyID','LastProgID','TopLeafID','DescendantID',
          'SnapNum','Redshift','Stars_Mass','SF_Mass','GroupID'
        ]

#Loop para todas las galaxias de la tabla
for p in np.arange(long):

    galid = df['GalaxyID'].to_numpy()[p]
 
    # Descargar todos los IDS necesarios de la galaxia deseada

    myIDs    = mtt.retrieve_ids(usr,pwd,simu,snap,galid)

    raw_tree = mtt.download_merger_tree(usr,pwd,simu,myIDs['GalaxyID'],myIDs['LastProgID'],
                                  table=table,table_alias=table_alias,columns=columns)
    
    # Aplicar condiciones a las galaxias del árbol, si es necesario
    mask = (
            (raw_tree['Stars_Mass']>0)
           )

    # Armar arbol sólo con galaxias seleccionadas según condiciones anteriores
    tree = {}
    for key in raw_tree.keys():
        tree[key]=raw_tree[key][mask]
        
    # Ordeno las galaxias según SnapNum creciente
    mask_order = (tree['SnapNum']).argsort()
    for key in tree.keys():
        tree[key] = tree[key][mask_order]

    # Armo un diccionario con solo la main branch del árbol
    main_branch = {}
    # Select galaxies in the main branch 
    mask_main = np.logical_and(tree['GalaxyID']>=myIDs['GalaxyID'],
                               tree['GalaxyID']<=myIDs['TopLeafID']) 

    for key in tree.keys():
        main_branch[key] = tree[key][mask_main]

        
    xplot_main = main_branch['SnapNum']
    
    # Calculo level of merger
    level_merger=[1]       # Inicializo con un 1, porque la primer galaxia del main branch 
                           # no viene de ninguna fusión...
    
    for k in range(np.size(xplot_main)-1):
        m1=main_branch['Stars_Mass'][k]+main_branch['SF_Mass'][k]
        mask=(tree['DescendantID']==main_branch['GalaxyID'][k+1]) & (tree['SnapNum']!=28) 
        # La condición de snapnum distinto a 28 es porque galaxias a z=0 no tienen descendiente,
        # y se les asigna como DescendantID su propio GalaxyID
        m2=np.sum(tree['Stars_Mass'][mask])+np.sum(tree['SF_Mass'][mask])
        level=m2/m1
        level_merger=np.append(level_merger,level)
        
    for snap, Lm_value in zip(main_branch['SnapNum'], level_merger):
        rows.append({
            "GalaxyID": galid,
            "Snapshot": int(snap),
            "Lm": float(Lm_value)
        })  
        

    
    print('Porcentaje:',round(float((p+1)/long),3),end='\r')


In [23]:
DATA_merge = pl.DataFrame(rows)
DATA_merge

GalaxyID,Snapshot,Lm
i64,i64,f64
1620592,3,1.0
1620592,4,1.0
1620592,5,1.0
1620592,6,2.777111
1620592,7,1.048094
…,…,…
3265053,24,1.088847
3265053,25,1.013717
3265053,26,1.114868


In [ ]:
# Guardamos el DataFrame con el historial de fusiones de las galaxias centrales
#DATA_merge.write_csv(path_lin+'historial_merger_centrales_NoAGNL0050N0752_snap28.csv')

In [42]:
mask = (pl.col('Snapshot') == 23) & (pl.col('Lm') > 1.25)& (pl.col('Lm') < 2.5)
DATA_merge.filter(mask)

GalaxyID,Snapshot,Lm
i64,i64,f64
1620592,23,2.323044
3163510,23,1.505014
268424,23,1.472859
3269820,23,1.590497
3290325,23,1.503967
…,…,…
3631727,23,1.289133
2619114,23,2.011477
2539050,23,1.321078


In [26]:
# Chusmeo la cantidad de fusiones que hay por snapshot, Lm>1.1 y Lm<2.0
mask = (pl.col('Snapshot')==12) & (pl.col('Lm')<1.1) & (pl.col('Lm')<2.5)
DATA_merge.filter(mask)

GalaxyID,Snapshot,Lm
i64,i64,f64
1620592,12,1.015702
3126435,12,1.005275
3131127,12,1.0
3136043,12,1.034777
3139793,12,1.012682
…,…,…
3251598,12,1.0
3255057,12,1.003493
3258553,12,1.005394


In [ ]:
# Fin